# notebook d'expérimentation ....
## objectifs
- analyser VOS données 
- charger et visualiser des spectres 2D ou 1D
- analyser les raies, ajuster des gaussiennes...

$\rightarrow$ **en cours :**
- TD L3 Rennes : version Yveline (avec scipy) + version Pascal (avec specutils + calcul du V/R)


In [37]:
%matplotlib widget
import numpy as np
from spectro_dashboard import SpectroDashboard

# 1. Afficher le dashboard
db = SpectroDashboard()
db.show()


In [38]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from astropy.modeling import models, fitting
from specutils import Spectrum, SpectralRegion
from astropy import units as u
from specutils.analysis import centroid, fwhm, equivalent_width, snr, snr_derived


In [39]:
### récupération de la vitesse barycentrique

import astropy.units as u
from astropy.time import Time
from astropy.coordinates import SkyCoord, EarthLocation
from astropy.constants import c

c_kms = c.to('km/s')      #c = 299792.458 km/s

### coord du CALC
obs_longitude = 1.5 * u.Unit('deg')
obs_latitude = 48.0 * u.Unit('deg')
obs_height = 50 * u.Unit('m')

obs_loc = EarthLocation(lat = obs_latitude, lon = obs_longitude, height = obs_height)
obs_time = Time('2025-09-30 23:00:00')

# on récupère ses coords (attention : syntaxe SIMBAD)
target_coord = SkyCoord.from_name('alcyone')
#target_coord = SkyCoord.from_name('pleione')
print(target_coord)

obs_coord = EarthLocation(lon = obs_longitude, lat = obs_latitude)

# vitesse héliocentrique
#heliocorr = target_coord.radial_velocity_correction('heliocentric', obstime=obs_time, location=obs_loc)
heliocorr = target_coord.radial_velocity_correction('barycentric', obstime=obs_time, location=obs_loc)
print(heliocorr.to(u.km / u.s))


<SkyCoord (ICRS): (ra, dec) in deg
    (56.8711523, 24.10513565)>
23.424773302874552 km / s


In [40]:
#base_dir = Path(__file__).resolve().parent
# corrections plouis : changement des chemins / fichiers
base_dir = Path('data/TP_Rennes')

# Choose spectrum
spectrum = [
            'Spectres/2026/_alcyone_20250930_392.dat',
           'Spectres/2026/_alp_cep_20250930_856.dat',
           'Spectres/2019/ngc6543_20191021_784.dat',
           'Spectres/2019/ngc6543_20191021_784_continuum_off.dat',
           'Spectres/2019/agdra_20191021_877.dat',
           'Spectres/2019/hd145454_20180924_918.dat',
           'Spectres/2026/_pleione_20250930_939.dat',
            ]

#print("Available spectra : alcyone (0), alp cep (1), ngc6543 (2), ngc6543 continuum (3), agdra (4), hd145454 (5), pleione (6)")
#i = int(input("Choose your spectrum: "))
i=0
file_path = base_dir/spectrum[i]

if not file_path.exists():
    raise(ValueError(f"Error: File not found: {file_path}"))
    
# on charge le spectre
_spc_array = np.loadtxt(file_path)
_spec1d = Spectrum(spectral_axis=_spc_array[:,0] * u.Unit('Angstrom'), flux=_spc_array[:,1] * u.Unit('mJy'))

# on décale de la vitesse barycentrique
_spec1d = Spectrum(spectral_axis=_spec1d.spectral_axis * (1 + heliocorr /c), flux=_spec1d.flux)

db.show_spectrum(_spec1d.wavelength, _spec1d.flux, label='corr_'+file_path.stem[1:5]+'...', color='red')


--> Spectre 'corr_alcy...' affiché : 5356 pts, X:[6510.5:6710.4]


In [41]:
# ATTENTION : le spectre autour de H alpha d'une étoile Be est un 'shell', i.e. une enveloppe de gaz autour de l'étoile
# -> le modèle à ajuster n'est PAS une double gaussienne mais une large gaussienne en émission, 'percée' d'une autre gaussienne en absorption

import numpy as np
import matplotlib.pyplot as plt
from astropy.modeling import models, fitting
import astropy.units as u
from astropy.time import Time
from astropy.coordinates import SkyCoord, EarthLocation

# on définit une zone autour des cibles
lambda_ha = 6562.82
lambda_HeI = 6678.15 
window_width = 20       # nb angstroms de chaque côté

# on prépare les données fixes pour l'incertitude systématique (instrument = starEx 2400)
fwhm_neon_pix = 5.0   # largeur raie néon en px, retourné par specinti
disp_physique = 0.0622 # dispersion du starEx2400, retourné par specinti
R_power = 19000 
fwhm_inst = lambda_ha / R_power # largeur instrumentale

# on utilise les valeurs brutes (pour éviter les soucis de numpy avec les unités astropy)
x_data = _spec1d.spectral_axis.value
y_data = _spec1d.flux.value 

# on découpe la zone 
mask = (x_data > lambda_ha - window_width) & (x_data < lambda_ha + window_width)
x_win = x_data[mask]
y_win = y_data[mask]

# on prépare les valeurs d'aide pour le fitter
y_min = np.min(y_win)
y_max = np.max(y_win)
y_cont_guess = y_min
amp_em_guess = y_max - y_min
amp_abs_guess = (y_max - y_min) * 0.5 # On suppose que l'absorption mange la moitié du shell

# on initialise les 3 modèles
model_init = (models.Gaussian1D(amplitude=amp_em_guess, mean=lambda_ha, stddev=2.5) +     # Emission large
              models.Gaussian1D(amplitude=amp_abs_guess, mean=lambda_ha, stddev=0.5) +    # Absorption fine
              models.Const1D(amplitude=1.0))                                              # Continuum

# on ajuste
fitter = fitting.LevMarLSQFitter()
fit_result = fitter(model_init, x_win, y_win)

# on extrait les données de la large gausssienne en emission
stddev_mesure = fit_result[0].stddev.value
fwhm_mesure = 2.355 * stddev_mesure      # FWHM = 2.355 * sigma (2.355 = 2 * sqrt(2 * ln 2) pour une gaussienne)

print("-" * 40)
print(f"FWHM Mesurée                     : {fwhm_mesure:.3f} +/- {stddev_mesure:.2f} A")

# on recherche maintenant les deux pics V et R sur la courbe fittée
# on prend les maxima de part et d'autre du centre estimé (halpha)
y_fit = fit_result(x_win)
mask_V = x_win < lambda_ha
mask_R = x_win > lambda_ha

# on trouve les index des maximum dans le masque
idx_V_relatif = np.argmax(y_fit[mask_V])
idx_R_relatif = np.argmax(y_fit[mask_R])

# on récupére la position (en Angströms)
x_V = x_win[mask_V][idx_V_relatif]
x_R = x_win[mask_R][idx_R_relatif]

# on récupère les intensité relatives
i_V = y_fit[mask_V][idx_V_relatif]
i_R = y_fit[mask_R][idx_R_relatif]

# on en déduit le V/R
v_sur_r = (i_V - 1) / (i_R - 1)

# et la vitesse de rotation du disque
delta_lambda = x_R - x_V
v_rot_disk = (c_kms * (delta_lambda / lambda_ha)) / 2

# on établit l'incertitude finale à partir du néon : on prend 1/10 de la FWHM d'une raie du néon
fwhm_inst_a = fwhm_neon_pix * disp_physique  # Résolution en Angströms
err_pos = fwhm_inst_a / 10.0                 

err_delta_lambda = np.sqrt(err_pos**2 + err_pos**2)     # on propage car il y a 2 pics
err_v = c * (err_delta_lambda / (2 * lambda_ha))         
    
print("-" * 40)
print(f"Pic Bleu (V)                     : {x_V:.3f} +/- {err_pos:.3f} A")
print(f"Pic Rouge (R)                    : {x_R:.3f} +/- {err_pos:.3f} A")
print(f"Séparation (dL)                  : {delta_lambda:.3f} +/- {err_delta_lambda:.03f} A")
print(f"Rapport V/R (asymétrie disque)   : {v_sur_r:.2f}")
print(f"Vitesse du gaz au bord disque    : {v_rot_disk.value:.1f} +/- {err_v.to('km/s').value:.1f} km/s")
print("-" * 40)

# on affiche le tout
x_plot = np.linspace(x_win.min(), x_win.max(), 1000)

db.clear_spectra()
db.show_spectrum(x_win, y_win, label='brut')
db.show_spectrum(x_win, y_fit, label='fit')

# on affiche les Lignes V/R
db.ax_spec.axvline(x_V, color='blue', linestyle='-', alpha=0.8, label=f'Centre V ({x_V:.2f} A)')
db.ax_spec.axvline(x_R, color='green', linestyle='-', alpha=0.8, label=f'Centre R ({x_R:.2f} A)')
db.ax_spec.legend()



----------------------------------------
FWHM Mesurée                     : 3.831 +/- 1.63 A
----------------------------------------
Pic Bleu (V)                     : 6562.092 +/- 0.031 A
Pic Rouge (R)                    : 6563.584 +/- 0.031 A
Séparation (dL)                  : 1.492 +/- 0.044 A
Rapport V/R (asymétrie disque)   : 1.00
Vitesse du gaz au bord disque    : 34.1 +/- 1.0 km/s
----------------------------------------
--> Spectre 'brut' affiché : 1072 pts, X:[6542.8:6582.8]
--> Spectre 'fit' affiché : 1072 pts, X:[6542.8:6582.8]


# ajustement de la raie HeI


In [42]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.modeling import models, fitting
from specutils import Spectrum
from astropy import units as u

R_power = 19000 
lambda_HeI = 6678.15 # Longueur d'onde au repose de He I

# Calcul de la largeur instrumentale : FWHM_inst = Lambda / R
fwhm_inst = lambda_HeI / R_power

# Chargement
#sp = Spectrum.read(filename)
sp = _spec1d
x_data = sp.spectral_axis.value
y_data = sp.flux.value

# on encadre heI
mask = (x_data > 6660) & (x_data < 6700)
x_window = x_data[mask]
y_window = y_data[mask]

# On cherche un continuum (constante) + une Gaussienne NÉGATIVE (trou)
y_cont_estim = np.max(y_window) # Le continuum est le haut du signal
y_dip_estim = np.min(y_window) - y_cont_estim # La profondeur du trou

# on initialize le modèle : Gaussienne (Absorption) + Constante
g_abs = models.Gaussian1D(amplitude=y_dip_estim, mean=lambda_HeI, stddev=2.0)
continuum = models.Const1D(amplitude=y_cont_estim)
model_he = g_abs + continuum

# on ajuste
fitter = fitting.LevMarLSQFitter()
fit = fitter(model_he, x_window, y_window)

# on récupère l'incertitude du fit gaussien
stddev_mesure = fit[0].stddev.value

# à partir de laquelle on extrait la FWHM (Cayrel)
fwhm_mesure = 2.355 * stddev_mesure

# Correction de la résolution instrumentale (moyenne quadratique)
fwhm_star = np.sqrt(fwhm_mesure**2 - fwhm_inst**2)

# Conversion en vitesse
# Le facteur 1.3 compense l'assombrissement centre-bord (Limb Darkening)
coeff_ld = 1.0 #1.3 
c = 299792.458
v_sin_i = (c * fwhm_star) / (lambda_HeI * coeff_ld)
err_v = c * (stddev_mesure / (2 * lambda_HeI))

# on affiche le tout
db.clear_spectra()

# spectre brut
db.show_spectrum(x_window, y_window, label='spectre')

# le Fit
db.show_spectrum(x_window, fit(x_window), label='Fit')

# et les résultats
print(f"FWHM Instrumentale           : {fwhm_inst:.3f} A")
print(f"FWHM Mesurée (Totale)        : {fwhm_mesure:.3f} +/- {stddev_mesure:.2f} A")
print(f"FWHM Étoile (Corrigée)       : {fwhm_star:.3f} +/- {stddev_mesure:.2f} A")
print("-" * 40)
print(f"VITESSE DE ROTATION (v sin i) : {v_sin_i:.0f} +/- {err_v:.0f} km/s")
print("-" * 40)

--> Spectre 'spectre' affiché : 1072 pts, X:[6660.0:6700.0]
--> Spectre 'Fit' affiché : 1072 pts, X:[6660.0:6700.0]
FWHM Instrumentale           : 0.351 A
FWHM Mesurée (Totale)        : 3.955 +/- 1.68 A
FWHM Étoile (Corrigée)       : 3.940 +/- 1.68 A
----------------------------------------
VITESSE DE ROTATION (v sin i) : 177 +/- 38 km/s
----------------------------------------


In [43]:
### calcul du rayon du disque / étoile 

R = (v_sin_i / v_rot_disk) ** 2

print("-" * 40)
print(f"RAYON DU DISQUE / étoile : {R.value:.0f} ")
print("-" * 40)



----------------------------------------
RAYON DU DISQUE / étoile : 27 
----------------------------------------
